# Potato Leaf Disease Classification using CNN

**Goal:** classify leaf images into **Early Blight, Late Blight, Healthy**.

Run `python prepare_data.py` from the project root **before** running this notebook.
This project uses **original images**, a separate test split, and a CNN built from scratch.

## 1. Imports and project paths

Check the selected VS Code notebook kernel is your `.venv`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, classification_report

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data" / "split"
MODEL_DIR = ROOT / "models"
REPORT_DIR = ROOT / "reports"
MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)
assert (DATA / "train").exists(), "Run python prepare_data.py from the project folder first"
print("TensorFlow:", tf.__version__)
print("Project folder:", ROOT)

## 2. Understand class distribution and look at raw images

Folders are the labels. Count each class before training to detect imbalance.

In [ ]:
class_names = sorted(p.name for p in (DATA / "train").iterdir() if p.is_dir())
class_counts = {c: len(list((DATA / "train" / c).glob("*.jpg"))) for c in class_names}
print("Class names in exact prediction order:", class_names)
print("Train counts:", class_counts)
fig, axes = plt.subplots(1, len(class_names), figsize=(12, 4))
for ax, class_name in zip(axes, class_names):
    sample = next((DATA / "train" / class_name).glob("*.jpg"))
    ax.imshow(Image.open(sample).convert("RGB"))
    ax.set_title(class_name.replace("Potato___", ""))
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Load train, validation and test images

`image_dataset_from_directory` reads the three class folders, assigns integer labels, resizes to **128×128**, and loads RGB images. Validation/test are not shuffled for repeatable evaluation.

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
SEED = 42

def make_dataset(split, shuffle=False):
    return tf.keras.utils.image_dataset_from_directory(
        DATA / split,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        color_mode="rgb",
        label_mode="int",
        class_names=class_names,
        shuffle=shuffle,
        seed=SEED,
    )

train_ds = make_dataset("train", shuffle=True)
val_ds = make_dataset("val")
test_ds = make_dataset("test")
print("One image batch:", next(iter(train_ds))[0].shape)

## 4. Augmentation: training images only

A leaf may rotate in a real photograph. Small, realistic variations can help a CNN generalize. This does **not** automatically equalize class counts.

Preprocessing choice: **Rescaling inside the saved model**. Input pixels enter as values from 0 to 255, and the model divides by 255.

In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
], name="train_only_augmentation")

sample_batch, _ = next(iter(train_ds))
sample = sample_batch[0]
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(sample.numpy().astype("uint8"))
axes[0].set_title("Original")
for i in range(1, 4):
    transformed = augmentation(tf.expand_dims(sample / 255.0, axis=0), training=True)[0]
    axes[i].imshow(tf.clip_by_value(transformed, 0, 1))
    axes[i].set_title("Augmented")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. CNN architecture (everything is visible here)

Conv2D learns patterns; MaxPool reduces dimensions; Flatten creates a feature vector; Dense combines features; Dropout helps regularize; Softmax gives three class scores.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(128, 128, 3)),
    tf.keras.layers.Rescaling(1.0 / 255),
    augmentation,
    tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(len(class_names), activation="softmax"),
], name="potato_cnn")
model.summary()

## 6. Compile and train

Three mutually exclusive classes → `SparseCategoricalCrossentropy`, because labels are integer IDs. **Class weights** compensate for unequal training examples (especially fewer Healthy images).

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
count_total = sum(class_counts.values())
class_weights = {
    i: count_total / (len(class_names) * class_counts[name])
    for i, name in enumerate(class_names)
}
print("Class weights:", class_weights)
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    class_weight=class_weights,
)

## 7. Plot loss and accuracy

Training and validation curves help identify learning progress and possible overfitting. More epochs do not always mean better performance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.history["accuracy"], label="Train")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(history.history["loss"], label="Train")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.savefig(REPORT_DIR / "learning_curves.png", dpi=140)
plt.show()

## 8. Evaluate held-out test set + confusion matrix

The confusion matrix shows *which* classes are mixed up, even if total accuracy looks good. Train for as long as needed using validation only; do not repeatedly tune to the test score.

In [ ]:
loss, accuracy = model.evaluate(test_ds, verbose=0)
print(f"Test loss: {loss:.4f} | Test accuracy: {accuracy:.2%}")
true_labels = np.concatenate([y.numpy() for _, y in test_ds])
probabilities = model.predict(test_ds, verbose=0)
predicted_labels = np.argmax(probabilities, axis=1)
print(classification_report(true_labels, predicted_labels, target_names=class_names, zero_division=0))
cm = confusion_matrix(true_labels, predicted_labels, labels=range(len(class_names)))
short_names = [name.replace("Potato___", "") for name in class_names]
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=short_names).plot(ax=ax, cmap="Blues", colorbar=False)
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(REPORT_DIR / "confusion_matrix.png", dpi=140)
plt.show()

## 9. Save model + label mapping

The saved model includes rescaling and training-only augmentation. **Never divide by 255 again** when making predictions in FastAPI.

In [ ]:
model.save(MODEL_DIR / "potato_cnn.keras")
(MODEL_DIR / "class_names.json").write_text(json.dumps(class_names, indent=2), encoding="utf-8")
print("Saved:", MODEL_DIR / "potato_cnn.keras")
print("Saved:", MODEL_DIR / "class_names.json")

## 10. Predict a completely unseen photo (optional)

Place a photo **not present in train, validation, or test** at `data/unseen/leaf.jpg` and run. A phone photo can have different lighting/background, so errors are possible.

In [ ]:
unseen_file = ROOT / "data" / "unseen" / "leaf.jpg"
if unseen_file.is_file():
    img = Image.open(unseen_file).convert("RGB").resize(IMG_SIZE)
    x = np.expand_dims(np.asarray(img, dtype=np.float32), axis=0)
    scores = model.predict(x, verbose=0)[0]
    best = int(np.argmax(scores))
    print("Predicted:", class_names[best])
    print("Model score (not a calibrated probability):", f"{scores[best]:.1%}")
    plt.imshow(img)
    plt.axis("off")
    plt.show()
else:
    print("Optional: add a truly new photo at data/unseen/leaf.jpg")

## What next?

Start `uvicorn api.main:app --reload` and `streamlit run frontend/app.py` in two separate terminals. The API loads the model and label mapping you saved.